# Reverse transfer: PhreshPhish -> PhiUSIIL (A2, frozen)

**Purpose.** The manuscript reports a frozen one-direction transfer PhiUSIIL -> PhreshPhish
(F1 0,599 for LightGBM; MCC -0,146). This experiment runs the mirror image so that the portability
claim is symmetric rather than a single anecdote.

**Design (mirrors Experiment 7 / Table 8 where the forward arm lives).**

| Component | Value |
|---|---|
| Training corpus | PhreshPhish **fit block only**, 2024-07-02 <= date < 2025-07-01 |
| Target corpus | cleaned PhiUSIIL, 235.370 rows (never seen by the model) |
| Representation | A2, 18 URL-only features, shared extractor from Experiment 5 |
| Model | LightGBM 300 estimators, lr 0,05, 63 leaves, seeds 13/42/777/2024/31415 |
| Threshold | 0,5 frozen; no calibration, no target tuning |
| Uncertainty | domain percentile bootstrap, 1.000 reps, RNG seed 42, unit = registrable domain |
| In-source reference | same model on the PhreshPhish test block (2025-09-08 <= date < 2025-12-17) |

**What this is not.** It is not a temporal evaluation. PhiUSIIL carries no collection dates, so the
target cannot be placed on a timeline. This is a frozen cross-corpus transfer in the opposite
direction.

In [1]:
import hashlib, importlib.util, json, platform, sys, time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import lightgbm as lgb
import sklearn
import pyarrow.parquet as pq
from sklearn.metrics import (accuracy_score, average_precision_score, confusion_matrix,
                             f1_score, matthews_corrcoef, precision_score, recall_score,
                             roc_auc_score)

WS = Path(r"E:\Kuliah\Tugas Akhir")
PHI = WS / "Experiment" / "1 - PhiUSIIL Cleaning + Leakage-Audit Notebook" / "phiusill_clean" / "phiusiil_clean.parquet"
PHRESH_DIR = WS / "dataset" / "PhreshPhish" / "data"
FEATURES_PY = WS / "Experiment" / "5 - PhiUSIIL Focused Ablation" / "src" / "features.py"
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%f") + "Z"
OUT = WS / "Experiment" / "9 - Reverse Transfer PhreshPhish to PhiUSIIL" / "outputs" / RUN_ID
OUT.mkdir(parents=True, exist_ok=True)

FIT_START = pd.Timestamp("2024-07-02")
VAL_START = pd.Timestamp("2025-07-01")
CUTOFF    = pd.Timestamp("2025-09-08")
TEST_END  = pd.Timestamp("2025-12-17")
SEEDS = [13, 42, 777, 2024, 31415]
PRIMARY_SEED = 42
THRESHOLD = 0.5
REPS = 1000
BOOT_SEED = 42
MODEL_PARAMS = dict(n_estimators=300, learning_rate=0.05, num_leaves=63, n_jobs=-1, verbose=-1)

spec = importlib.util.spec_from_file_location("shared_features", FEATURES_PY)
fx = importlib.util.module_from_spec(spec); spec.loader.exec_module(fx)

def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()

print("run id      :", RUN_ID)
print("output dir  :", OUT)
print("python      :", platform.python_version(), "| pandas", pd.__version__, "| numpy", np.__version__,
      "| lightgbm", lgb.__version__, "| sklearn", sklearn.__version__)
print("extractor   :", FEATURES_PY.name, sha256(FEATURES_PY)[:16], "| features:", len(fx.FEATURE_ORDER))

run id      : 20261101T062209564437Z
output dir  : E:\Kuliah\Tugas Akhir\Experiment\9 - Reverse Transfer PhreshPhish to PhiUSIIL\outputs\20261101T062209564437Z
python      : 3.14.7 | pandas 3.1.0 | numpy 2.5.3 | lightgbm 4.7.0 | sklearn 1.9.1
extractor   : features.py 81822b4fac8b50cd | features: 18


In [2]:
t0 = time.perf_counter()
phi = pd.read_parquet(PHI, columns=["URL", "label"])
# Source polarity: 1 = legitimate, 0 = phishing. Analysis convention: phishing = 1.
phi["y"] = (phi["label"] == 0).astype("int8")
phi = phi.rename(columns={"URL": "url"})
print("PhiUSIIL cleaned rows :", len(phi))
print("label 1 (benign)      :", int((phi.label == 1).sum()), "| label 0 (phishing):", int((phi.label == 0).sum()))
print("analysis y_phish=1    :", int(phi.y.sum()), "| y=0:", int((phi.y == 0).sum()))
print(f"loaded in {time.perf_counter()-t0:.1f}s")

PhiUSIIL cleaned rows : 235370
label 1 (benign)      : 134850 | label 0 (phishing): 100520
analysis y_phish=1    : 100520 | y=0: 134850
loaded in 0.1s


In [3]:
t0 = time.perf_counter()
train_files = sorted(PHRESH_DIR.glob("train-*.parquet"))
test_files  = sorted(PHRESH_DIR.glob("test-*.parquet"))
print("train shards:", len(train_files), "| test shards:", len(test_files))

def load_block(files, start, end):
    parts, rows = [], 0
    for f in files:
        d = pq.read_table(f, columns=["url", "label", "date"]).to_pandas()
        rows += len(d)
        d["date"] = pd.to_datetime(d["date"])
        parts.append(d[(d.date >= start) & (d.date < end)])
    out = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()
    return out, rows

fit_raw, train_total = load_block(train_files, FIT_START, VAL_START)
test_raw, test_total = load_block(test_files, CUTOFF, TEST_END)
print(f"rows read: train shards {train_total:,} | test shards {test_total:,}")
print(f"fit block raw        : {len(fit_raw):,}  ({fit_raw.date.min().date()} .. {fit_raw.date.max().date()})")
print(f"test block raw       : {len(test_raw):,}  ({test_raw.date.min().date()} .. {test_raw.date.max().date()})")

fit = fit_raw.drop_duplicates(subset="url", keep="first").copy()
test = test_raw.drop_duplicates(subset="url", keep="first").copy()
fit["y"] = (fit.label == "phish").astype("int8")
test["y"] = (test.label == "phish").astype("int8")
print(f"fit after url dedup  : {len(fit):,}  (dropped {len(fit_raw)-len(fit):,}) | phishing {int(fit.y.sum()):,}")
print(f"test after url dedup : {len(test):,}  (dropped {len(test_raw)-len(test):,}) | phishing {int(test.y.sum()):,}")
print("label domain check   :", sorted(fit.label.unique()), sorted(test.label.unique()))
print(f"loaded in {time.perf_counter()-t0:.1f}s")

train shards: 56 | test shards: 21


rows read: train shards 498,255 | test shards 168,060
fit block raw        : 372,381  (2024-07-02 .. 2025-06-30)
test block raw       : 168,060  (2025-09-08 .. 2025-12-16)


fit after url dedup  : 372,381  (dropped 0) | phishing 190,618
test after url dedup : 168,060  (dropped 0) | phishing 76,800
label domain check   : ['benign', 'phish'] ['benign', 'phish']
loaded in 1.2s


In [4]:
t0 = time.perf_counter()

def extract(urls, name):
    rows, bad = [], 0
    for u in urls:
        try:
            rows.append(fx.url_features(u))
        except Exception:
            bad += 1
            rows.append({k: np.nan for k in fx.FEATURE_ORDER})
    X = pd.DataFrame(rows, columns=fx.FEATURE_ORDER, dtype="float64")
    ok = ~X.isna().any(axis=1)
    print(f"{name:22s} rows {len(X):,} | unparseable {bad:,} | usable {int(ok.sum()):,}")
    return X, ok

X_fit,  ok_fit  = extract(fit.url.tolist(),  "PhreshPhish fit")
X_test, ok_test = extract(test.url.tolist(), "PhreshPhish test")
X_phi,  ok_phi  = extract(phi.url.tolist(),  "PhiUSIIL target")
print(f"extraction total {time.perf_counter()-t0:.1f}s")

PhreshPhish fit        rows 372,381 | unparseable 1 | usable 372,380


PhreshPhish test       rows 168,060 | unparseable 0 | usable 168,060


PhiUSIIL target        rows 235,370 | unparseable 0 | usable 235,370
extraction total 59.2s


In [5]:
fit, X_fit   = fit[ok_fit.values].reset_index(drop=True),   X_fit[ok_fit.values].reset_index(drop=True)
test, X_test = test[ok_test.values].reset_index(drop=True), X_test[ok_test.values].reset_index(drop=True)
phi, X_phi   = phi[ok_phi.values].reset_index(drop=True),   X_phi[ok_phi.values].reset_index(drop=True)
print("aligned sizes:", {"fit": len(fit), "test": len(test), "phi": len(phi)})

# Overlap audit: the model must never have seen the target corpus.
fit_urls, phi_urls = set(fit.url), set(phi.url)
fit_dom = {fx.domain_for(u) for u in fit.url}
phi_dom = {fx.domain_for(u) for u in phi.url}
overlap = pd.DataFrame([
    dict(measure="exact_url", n_train=len(fit_urls), n_target=len(phi_urls),
         n_common=len(fit_urls & phi_urls), pct_target_rows=100*phi.url.isin(fit_urls).mean()),
    dict(measure="registrable_domain", n_train=len(fit_dom), n_target=len(phi_dom),
         n_common=len(fit_dom & phi_dom), pct_target_rows=100*phi.url.map(fx.domain_for).isin(fit_dom).mean()),
])
print(overlap.to_string(index=False))

aligned sizes: {'fit': 372380, 'test': 168060, 'phi': 235370}


           measure  n_train  n_target  n_common  pct_target_rows
         exact_url   372380    235370       137         0.058206
registrable_domain   102352    175509      9026        21.611930


In [6]:
results, confusions, preds = [], [], {}
t0 = time.perf_counter()
for seed in SEEDS:
    model = lgb.LGBMClassifier(**MODEL_PARAMS, random_state=seed)
    t_fit = time.perf_counter()
    model.fit(X_fit, fit.y)
    fit_s = time.perf_counter() - t_fit
    for arm, X, frame in (("target_PhiUSIIL", X_phi, phi), ("in_source_PhreshPhish", X_test, test)):
        p = model.predict_proba(X)[:, 1]
        yhat = (p >= THRESHOLD).astype("int8")
        tn, fp, fn, tp = confusion_matrix(frame.y, yhat, labels=[0, 1]).ravel()
        results.append(dict(seed=seed, arm=arm, n=len(frame), n_benign=int((frame.y == 0).sum()),
                            n_phish=int((frame.y == 1).sum()), tn=int(tn), fp=int(fp), fn=int(fn), tp=int(tp),
                            f1_phish=f1_score(frame.y, yhat, pos_label=1, zero_division=0),
                            precision_phish=precision_score(frame.y, yhat, pos_label=1, zero_division=0),
                            recall_phish=recall_score(frame.y, yhat, pos_label=1, zero_division=0),
                            fpr_benign=fp / max(tn + fp, 1),
                            mcc=matthews_corrcoef(frame.y, yhat),
                            roc_auc=roc_auc_score(frame.y, p),
                            average_precision=average_precision_score(frame.y, p),
                            accuracy=accuracy_score(frame.y, yhat),
                            fit_s=fit_s))
        if seed == PRIMARY_SEED:
            preds[arm] = (p, yhat)
    r = [x for x in results if x["seed"] == seed]
    print(f"seed {seed:5d} fit {fit_s:5.2f}s | target F1 {r[0]['f1_phish']:.6f} FPR {r[0]['fpr_benign']:.6f} "
          f"MCC {r[0]['mcc']:+.6f} | in-source F1 {r[1]['f1_phish']:.6f} FPR {r[1]['fpr_benign']:.6f}")

per_seed = pd.DataFrame(results)
per_seed.to_csv(OUT / "metrics_per_seed.csv", index=False)
summary = (per_seed.groupby("arm")[["f1_phish", "precision_phish", "recall_phish", "fpr_benign", "mcc",
                                    "roc_auc", "average_precision", "accuracy"]]
           .agg(["mean", "std"]).round(9))
summary.to_csv(OUT / "metrics_summary.csv")
print(f"\nall fits in {time.perf_counter()-t0:.1f}s")
print(summary[["f1_phish", "fpr_benign", "mcc"]].to_string())

seed    13 fit  8.50s | target F1 0.551298 FPR 0.893267 MCC -0.083136 | in-source F1 0.805913 FPR 0.076277


seed    42 fit 10.10s | target F1 0.549564 FPR 0.897783 MCC -0.092320 | in-source F1 0.805958 FPR 0.077263


seed   777 fit  8.20s | target F1 0.550021 FPR 0.896270 MCC -0.089576 | in-source F1 0.805855 FPR 0.076079


seed  2024 fit  8.22s | target F1 0.550174 FPR 0.897849 MCC -0.090660 | in-source F1 0.805763 FPR 0.076320


seed 31415 fit 10.03s | target F1 0.550036 FPR 0.897011 MCC -0.090245 | in-source F1 0.805627 FPR 0.075816

all fits in 66.5s
                       f1_phish           fpr_benign                 mcc          
                           mean       std       mean       std      mean       std
arm                                                                               
in_source_PhreshPhish  0.805823  0.000132   0.076351  0.000547  0.677771  0.000195
target_PhiUSIIL        0.550219  0.000646   0.896436  0.001885 -0.089188  0.003531


In [7]:
# Domain percentile bootstrap on the PhiUSIIL target, primary seed, mirroring the manuscript protocol.
p, yhat = preds["target_PhiUSIIL"]
y = phi.y.to_numpy()
dom = phi.url.map(fx.domain_for).to_numpy()
codes, uniques = pd.factorize(pd.Series(dom))
n_dom = len(uniques)

tp = np.bincount(codes, weights=((y == 1) & (yhat == 1)).astype(float), minlength=n_dom)
fp = np.bincount(codes, weights=((y == 0) & (yhat == 1)).astype(float), minlength=n_dom)
fn = np.bincount(codes, weights=((y == 1) & (yhat == 0)).astype(float), minlength=n_dom)
tn = np.bincount(codes, weights=((y == 0) & (yhat == 0)).astype(float), minlength=n_dom)

rng = np.random.default_rng(BOOT_SEED)
rows = []
for _ in range(REPS):
    idx = rng.integers(0, n_dom, n_dom)
    TP, FP, FN, TN = tp[idx].sum(), fp[idx].sum(), fn[idx].sum(), tn[idx].sum()
    f1 = 2*TP / (2*TP + FP + FN) if (2*TP + FP + FN) else np.nan
    rec = TP / (TP + FN) if (TP + FN) else np.nan
    prec = TP / (TP + FP) if (TP + FP) else np.nan
    fpr = FP / (FP + TN) if (FP + TN) else np.nan
    den = np.sqrt((TP+FP)*(TP+FN)*(TN+FP)*(TN+FN))
    mcc = (TP*TN - FP*FN) / den if den else np.nan
    rows.append(dict(f1_phish=f1, recall_phish=rec, precision_phish=prec, fpr_benign=fpr, mcc=mcc))

boot = pd.DataFrame(rows)
point = per_seed[(per_seed.seed == PRIMARY_SEED) & (per_seed.arm == "target_PhiUSIIL")].iloc[0]
boot_out = pd.DataFrame([
    dict(metric=m, estimate=float(point[m]), lower95=float(boot[m].quantile(0.025)),
         upper95=float(boot[m].quantile(0.975)), valid_replicates=int(boot[m].notna().sum()),
         replicates=REPS, seed=BOOT_SEED, unit="registrable_domain", n_domains=n_dom)
    for m in ("f1_phish", "precision_phish", "recall_phish", "fpr_benign", "mcc")
])
boot_out.to_csv(OUT / "bootstrap_target_primary_seed.csv", index=False)
print(f"target bootstrap: {n_dom:,} domains, {REPS} replicates, RNG seed {BOOT_SEED}")
print(boot_out.to_string(index=False))

target bootstrap: 175,509 domains, 1000 replicates, RNG seed 42
         metric  estimate   lower95   upper95  valid_replicates  replicates  seed               unit  n_domains
       f1_phish  0.549564  0.512634  0.586764              1000        1000    42 registrable_domain     175509
precision_phish  0.409503  0.368570  0.455654              1000        1000    42 registrable_domain     175509
   recall_phish  0.835237  0.798916  0.869975              1000        1000    42 registrable_domain     175509
     fpr_benign  0.897783  0.895279  0.900180              1000        1000    42 registrable_domain     175509
            mcc -0.092320 -0.137850 -0.042795              1000        1000    42 registrable_domain     175509


In [8]:
design = dict(
    experiment="9 - reverse transfer PhreshPhish -> PhiUSIIL",
    direction="PhreshPhish(fit block) -> PhiUSIIL",
    is_temporal=False,
    note="PhiUSIIL has no collection dates, so this is a frozen cross-corpus transfer, not a temporal evaluation",
    training_corpus="PhreshPhish", evaluation_corpus="PhiUSIIL",
    split=dict(fit_start=str(FIT_START.date()), fit_end_exclusive=str(VAL_START.date()),
               in_source_test_start=str(CUTOFF.date()), in_source_test_end_exclusive=str(TEST_END.date())),
    representation="A2 (18 URL-only features)",
    extractor=dict(path=str(FEATURES_PY), sha256=sha256(FEATURES_PY), features=list(fx.FEATURE_ORDER)),
    label_mapping=dict(PhiUSIIL="1=legitimate -> y_phish=0 ; 0=phishing -> y_phish=1",
                       PhreshPhish="phish -> 1 ; benign -> 0"),
    model=dict(family="lightgbm", **MODEL_PARAMS), seeds=SEEDS, primary_seed=PRIMARY_SEED,
    threshold=dict(value=THRESHOLD, rule="score >= t", frozen=True, calibrated_on_target=False),
    bootstrap=dict(type="percentile", unit="registrable_domain", replicates=REPS, seed=BOOT_SEED),
    rows=dict(train_fit=int(len(fit)), in_source_test=int(len(test)), target=int(len(phi))),
    environment=dict(python=platform.python_version(), pandas=pd.__version__, numpy=np.__version__,
                     lightgbm=lgb.__version__, sklearn=sklearn.__version__,
                     platform=platform.platform()),
    run_id=RUN_ID,
)
(OUT / "design.json").write_text(json.dumps(design, indent=2, ensure_ascii=False), encoding="utf8")

inputs = [dict(file="phiusiil_clean.parquet", path=str(PHI), sha256=sha256(PHI), rows=int(len(phi)),
               role="target corpus (cleaned)"),
          dict(file="PhreshPhish train shards", path=str(PHRESH_DIR), sha256="see per-file manifest",
               rows=int(train_total), role="source corpus, fit block selected by date"),
          dict(file="features.py", path=str(FEATURES_PY), sha256=sha256(FEATURES_PY),
               rows=None, role="shared A2 extractor")]
(OUT / "input_manifest.json").write_text(json.dumps(inputs, indent=2), encoding="utf8")

overlap.to_csv(OUT / "overlap_train_vs_target.csv", index=False)

outputs = []
for f in sorted(OUT.glob("*")):
    if f.name != "output_manifest.json" and f.is_file():
        outputs.append(dict(file=f.name, bytes=f.stat().st_size, sha256=sha256(f)))
(OUT / "output_manifest.json").write_text(json.dumps(outputs, indent=2), encoding="utf8")

print("run written to", OUT)
for o in outputs:
    print(f"  {o['bytes']:>9,}  {o['file']}")

run written to E:\Kuliah\Tugas Akhir\Experiment\9 - Reverse Transfer PhreshPhish to PhiUSIIL\outputs\20261101T062209564437Z
        625  bootstrap_target_primary_seed.csv
      2,253  design.json
        851  input_manifest.json
      2,511  metrics_per_seed.csv
        693  metrics_summary.csv
        157  overlap_train_vs_target.csv


## Reading the result

Compare the `target_PhiUSIIL` row with the forward arm reported in the manuscript (Table 7,
PhiUSIIL -> PhreshPhish: F1 0,599064, FPR 0,986478, MCC -0,146445) and with the `in_source_PhreshPhish`
row produced here, which is the same model on data from its own corpus.

If the reverse arm also collapses, portability failure is bidirectional. If it does not, the failure is
asymmetric and the discussion has to say so. Either way this is transfer evidence, not temporal evidence.